# 2. Training (GPU T4 x2): English on GPU 0, Telugu on GPU 1 in parallel

Kaggle settings: **Accelerator: GPU T4 x2**, **Internet: On**.
**Add Input -> Your Work ->** `fastspeech2-1-data` (the data notebook's output).
To resume a previous training run, also add that training notebook's output.
Run with **Save Version -> Save & Run All (Commit)**. `MAX_HOURS` stops training cleanly
before Kaggle's 12 h limit; the last cells write plots, audio and the fp16 export.
Training progress (one line per 100 steps per language) streams to the log.

In [ ]:
# DRY_RUN = True: 300 steps per language + the full results stage, to verify every
# deliverable is produced before spending a 10.5 h session. Set False for the real run.
DRY_RUN = False
MAX_HOURS = 0.5 if DRY_RUN else 10.5

In [ ]:
import os, subprocess, glob, shutil, json, sys, time
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
subprocess.run(["git", "pull"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.environ["TQDM_MININTERVAL"] = "60"   # keep progress bars from flooding the log
os.environ["PYTHONUNBUFFERED"] = "1"


def sh(cmd):
    """Run a bash command, stream its output live, raise if it fails."""
    print("$", cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -euo pipefail; " + cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit {}): {}".format(p.returncode, cmd))
    print("[ok in {:.1f} min]".format((time.time() - t0) / 60), flush=True)

In [ ]:
sh("pip install -q -r requirements.txt")
import torch
assert torch.cuda.is_available(), "No CUDA GPU visible: set the notebook accelerator to GPU"
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    print(i, torch.cuda.get_device_name(i), "sm_%d%d" % cap)
    # fail fast if this torch build has no kernels for the GPU (e.g. P100 = sm_60)
    x = torch.randn(256, 256, device=f"cuda:{i}"); (x @ x).sum().item()
print("torch", torch.__version__, "cuda", torch.version.cuda, "arch list", torch.cuda.get_arch_list())

In [ ]:
if DRY_RUN:
    import yaml
    for lang in ["English", "Telugu"]:
        p = f"{lang}/config.yaml"
        c = yaml.safe_load(open(p, encoding="utf-8"))
        c["train"]["step"].update(total_step=300, log_step=50, val_step=100, save_step=300,
                                  latest_step=100)
        yaml.safe_dump(c, open(p, "w", encoding="utf-8"), allow_unicode=True)
    print("DRY RUN: 300 steps per language")

In [ ]:
# Link the prepared features from the data notebook output. Kaggle's mount layout for
# notebook outputs varies (/kaggle/input/<slug>/... or deeper), so search recursively.
sh("find /kaggle/input -maxdepth 6 -type d -name preprocessed | head -20")
for lang in ["English", "Telugu"]:
    src = sorted(glob.glob(f"/kaggle/input/**/data/{lang}/preprocessed", recursive=True))
    assert src, f"attach the data notebook output ({lang} preprocessed not found)"
    print(lang, "features from", src[0])
    os.makedirs(f"data/{lang}", exist_ok=True)
    if not os.path.exists(f"data/{lang}/preprocessed"):
        os.symlink(src[0], f"data/{lang}/preprocessed")
    res = os.path.dirname(os.path.dirname(os.path.dirname(src[0]))) + f"/Results/{lang}"
    for f in glob.glob(res + "/*.json") + glob.glob(res + "/*.tsv") + glob.glob(res + "/*.txt"):
        shutil.copy(f, f"Results/{lang}/")
    # resume: copy checkpoints and logs from a previous training run, if attached
    prev = sorted(glob.glob(f"/kaggle/input/**/{lang}/checkpoints/latest.pth.tar", recursive=True))
    if prev and not os.path.exists(f"{lang}/checkpoints/latest.pth.tar"):
        d = os.path.dirname(prev[-1])
        for f in glob.glob(d + "/*.pth.tar"):
            shutil.copy(f, f"{lang}/checkpoints/")
        shutil.copytree(d.replace(f"{lang}/checkpoints", f"Results/{lang}/logs"),
                        f"Results/{lang}/logs", dirs_exist_ok=True)
        print(lang, "resuming from", prev[-1])
    print(lang, json.load(open(f"data/{lang}/preprocessed/summary.json"))["train_items"], "train items")

In [ ]:
sh(f"bash scripts/run_pipeline.sh train-both {MAX_HOURS}")
for lang in ["English", "Telugu"]:
    assert os.path.exists(f"{lang}/checkpoints/best.pth.tar"), \
        f"{lang}: no best checkpoint, see Results/{lang}/logs/train_stdout.log"

In [ ]:
# loss plots, fp16 export, test-sentence audio + variance-adaptor plots
sh("bash scripts/run_pipeline.sh results english")
sh("bash scripts/run_pipeline.sh results telugu")
# keep the output small: drop numbered checkpoints, keep latest + best
sh("rm -f English/checkpoints/[0-9]*.pth.tar Telugu/checkpoints/[0-9]*.pth.tar; du -sh English/checkpoints Telugu/checkpoints Results")

In [ ]:
# verify every deliverable exists
problems = []
for lang, ids in [("English", ["E1", "E2a", "E2b", "E3", "E4"]), ("Telugu", ["T1", "T2a", "T2b", "T3", "T4"])]:
    R = f"Results/{lang}"
    for f in ["training_loss.png", "validation_loss.png", "loss_summary.json"]:
        if not os.path.exists(f"{R}/{f}"): problems.append(f"{R}/{f}")
    for i in ids:
        if not os.path.exists(f"{R}/test_generated_audios/{i}.wav"): problems.append(f"{lang} wav {i}")
        for k in ["duration", "pitch", "energy"]:
            if not os.path.exists(f"{R}/variance_adaptor_plots/{k}/{i}.png"): problems.append(f"{lang} {k} {i}")
    if not glob.glob(f"{lang}/checkpoints/*_infer_fp16.pth.tar"): problems.append(f"{lang} fp16 export")
    print(lang, json.load(open(f"{R}/loss_summary.json")))
assert not problems, problems
print("ALL DELIVERABLES PRESENT", "(dry run)" if DRY_RUN else "")